In [1]:
# === Imports (один раз в начале ноутбука) ===
import sys, torch
import sys, importlib
from snn_mnist_net import Cfg, run_experiment, save_snn, load_weights_into, build_net,build_encoder_from_cfg
from label_map import build_label_map,save_label_map, load_label_map
from counts_readout import collect_counts_plus_cuda, make_mnist_datasets
from readout_models import tfidf_from_counts, train_mlp_readout
# (опционально) from evaluation import evaluate_on_mnist

_MODS = [
    "snn_mnist_net",
    "label_map",
    "counts_readout",
    "readout_models",
]

# 1) вычистим из sys.modules сам модуль и подпакеты
for m in list(sys.modules):
    if m in _MODS or any(m.startswith(x + ".") for x in _MODS):
        sys.modules.pop(m, None)

# 2) инвалидируем кэши файловой системы
importlib.invalidate_caches()

# 3) заново импортируем модули как модули
import snn_mnist_net, label_map, counts_readout, readout_models

In [2]:

# LATENCY config
# === FULL RUN CONFIG (адаптировано под Cfg) ===
cfg = Cfg(
    # базовые
    time=200, n_hidden=100, device="cpu", seed=42,
    # обучение
    N=60000,                 # стартовый «полный» прогон
    log_every=500,
    # STDP
    nu_plus=1e-4, nu_minus=-1e-3,
    # ингибиция / WTA
    enable_inhibition_at_start=True,
    top_k=3,
    inhib_strength=0.705,
    # веса
    w_init_lo=0.25, w_init_hi=0.80,
    w_clip_min=0.0, w_clip_max=1.0,
    # пороги/динамика LIF
    thresh_init=0.38, tau_val=150.0, refrac_val=2.0, reset_val=0.0,
    # кодировщик
    encoder="latency", poisson_rate_scale=0.011,
    # прочее
    debug=True,
    warmup_N = 100,
    thresh_min = 0.05,
    bootstrap_threshold = 0.12,
    rest_val = 0.0,
    latency_x_min = 0,
    encoder_rate_boost = 1
    
)
print("FULL RUN:", {k: getattr(cfg, k) for k in [
    "inhib_strength","w_init_lo","w_init_hi","poisson_rate_scale","thresh_init","N"
]})




FULL RUN: {'inhib_strength': 0.705, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'poisson_rate_scale': 0.011, 'thresh_init': 0.38, 'N': 60000}


In [3]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

print("SUMMARY:", summary)
save_snn("out/snn_mnist_v6_latency_allpixels.pt", cfg, FF, lif_layer)

Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=60000
n_hidden=100  thresh_init=0.38
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.25, 0.8], clip=[0.0,1.0]
LIF: rest_val=0.0  reset_val=0.0  tau=150.0  refrac=2.0
Encoder=latency  poisson_rate_scale=0.011  base_seed=123
Inhibition: enable=True  inhib_strength=0.705  top_k=3
---------------------------

[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] rest  = scalar 0.000
[lif] dt = scalar 1.000


Warmup (T=200):   0%|                                                       | 0/100 [00:00<?, ?it/s]

[enc] shape=(200, 1, 784) sum=784.0 max=1.0
[enc] per_t: min=0.0 max=618.0 t_argmax=199


Warmup (T=200):   1%|▏                  | 1/100 [00:00<00:23,  4.16it/s, avg_rate=0.230 target=1.50]

[mon] in_sum=784.0 lif_sum=4600.0 v_max=0.0 vt_min=0.11999999731779099
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=784.0 max=1.0
[enc] per_t: min=0.0 max=608.0 t_argmax=199


...

Train (N=60000, T=200):   0%| | 2/60000 [00:00<3:45:03,  4.44it/s, spikes=600.00 uniq=6 HHI=0.167 e≈

[W] mean=0.524697 max=0.799988 min=0.243280
[enc] shape=(200, 1, 784) sum=784.0 max=1.0
[enc] per_t: min=0.0 max=664.0 t_argmax=199


Train (N=60000, T=200):   0%| | 3/60000 [00:00<4:30:02,  3.70it/s, spikes=600.00 uniq=9 HHI=0.111 e≈

[W] mean=0.524518 max=0.799988 min=0.242825


Train (N=60000, T=200):  33%|▎| 19518/60000 [1:07:31<2:11:33,  5.13it/s, spikes=462.71 uniq=100 HHI=IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Train (N=60000, T=200):  93%|▉| 55885/60000 [2:44:58<09:43,  7.05it/s, spikes=320.34 uniq=100 HHI=0.IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [4]:
# check latency network
print("SUMMARY:", summary)

SUMMARY: {'time': 200, 'n_hidden': 100, 'N': 60000, 'seed': 42, 'device': 'cpu', 'log_every': 500, 'debug': True, 'nu_plus': 0.0001, 'nu_minus': -0.001, 'enable_inhibition_at_start': True, 'inhib_strength': 0.705, 'encoder': 'latency', 'poisson_rate_scale': 0.011, 'poisson_base_seed': 123, 'thresh_init': 0.38, 'thresh_min': 0.05, 'thresh_max': 1.2, 'target_spikes': 1.5, 'ema_alpha': 0.9, 'ema_k': 0.02, 'tau_val': 150.0, 'refrac_val': 2.0, 'reset_val': 0.0, 'rest_val': 0.0, 'w_init_lo': 0.25, 'w_init_hi': 0.8, 'w_clip_min': 0.0, 'w_clip_max': 1.0, 'wmin': 0.0, 'wmax': 1.0, 'warmup_N': 100, 'top_k': 3, 'latency_x_min': 0, 'strong_inh_matrix': True, 'strong_inh_value': 0.65, 'bootstrap_threshold_enable': True, 'bootstrap_threshold': 0.12, 'bootstrap_refrac': 2.0, 'encoder_out_format': 'auto', 'poisson_deterministic': False, 'encoder_rate_boost': 1, 'spikes_per_sample': 311.22, 'synops_per_sample': 78400.0, 'v_updates_per_sample': 20000.0, 'energy_proxy_per_sample': 4331.22, 'winners_uniqu

In [6]:
import os
import importlib
import evaluation 
from evaluation import probe_readouts_counts
importlib.invalidate_caches()
importlib.reload(snn_mnist_net)
importlib.reload(evaluation)
importlib.invalidate_caches()
net, input_layer, lif_layer, connection, recurrent_inh, _ = build_net(cfg)
enc = build_encoder_from_cfg(cfg)

# 2) загрузить веса и пороги
load_weights_into(net, connection, lif_layer, "out/snn_mnist_v6_latency.pt")


[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] rest  = scalar 0.000
[lif] dt = scalar 1.000
Loaded from out/snn_mnist_v6_latency.pt


In [4]:
label_map_build = build_label_map(net, None, lif_layer, enc, n_calib=2000, T=cfg.time, top_k=3, seed=123)

save_label_map("out/label_v6_latency.pt",label_map_build, meta={"ckpt":"out/snn_mnist_v6_latency.pt","T":cfg.time})

Label-map built: 100/100 neurons assigned; active winners 100
[label_map] saved: out/label_v6_latency.pt  shape=(100,)


In [7]:
label_map_build = load_label_map("out/label_v6_latency.pt")

[label_map] loaded: out/label_v6_latency.pt  shape=(100,)  meta={'created_at': '2026-03-11 18:15:19', 'ckpt': 'out/snn_mnist_v6_latency.pt', 'T': 200}


In [8]:
from counts_readout import make_mnist_datasets

ds_train, ds_test = make_mnist_datasets()

In [14]:
from evaluation import eval_readouts_from_net
enc = build_encoder_from_cfg(cfg)
accs = eval_readouts_from_net(net, lif_layer, enc, cfg, label_map=label_map_build)
print(accs)   # ждём, что хотя бы counts_zscore+Linear или PCAwhiten+Linear > 0.6

[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–


[collect_counts_plus_cuda] offset summary: requested=+0.000, applied=0, avgΔvt=0.000, restore_ok=–
{'counts_zscore+Linear': 0.21809999644756317, 'PCAwhiten+Linear': 0.21619999408721924, 'TFIDF+MLP': 0.2718000113964081}


In [10]:
print(accs)

{'counts_zscore+Linear': 0.21969999372959137, 'PCAwhiten+Linear': 0.21610000729560852, 'TFIDF+MLP': 0.27000001072883606}


# Test Poisson

In [ ]:
import importlib, snn_mnist_net
importlib.reload(snn_mnist_net)
# === 1) Обучение SNN (STDP) ===
cfg.encoder = "poisson"
summary, FF, lif_layer, net, encoder = run_experiment(cfg, verbose=True, progress=True)
print("SUMMARY:", summary)

print("SUMMARY:", summary)
save_snn("out/snn_mnist_v6_poisson.pt", cfg, FF, lif_layer)

Используем: cpu

SNN-MNIST TRAIN START
seed=42  device=cpu  T=200  N=60000
n_hidden=100  thresh_init=0.38
STDP: nu_plus=0.0001  nu_minus=-0.001
FF init: w∈[0.25, 0.8], clip=[0.0,1.0]
LIF: rest_val=0.0  reset_val=0.0  tau=150.0  refrac=2.0
Encoder=poisson  poisson_rate_scale=0.011  base_seed=123
Inhibition: enable=True  inhib_strength=0.705  top_k=3
---------------------------

[lif] thresh = tensor mean=0.380 std=0.000 shape=(100,)
[lif] tc_decay = scalar 150.000
[lif] refrac = scalar 2.000
[lif] reset = scalar 0.000
[lif] rest  = scalar 0.000
[lif] dt = scalar 1.000


Warmup (T=200):   1%|▏                  | 1/100 [00:00<00:13,  7.30it/s, avg_rate=0.285 target=1.50]

[enc] shape=(200, 1, 784) sum=238.0 max=1.0
[enc] per_t: min=0.0 max=5.0 t_argmax=82
[mon] in_sum=238.0 lif_sum=5700.0 v_max=0.0 vt_min=0.11999999731779099
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=270.0 max=1.0
[enc] per_t: min=0.0 max=6.0 t_argmax=23


Warmup (T=200):   3%|▌                  | 3/100 [00:00<00:13,  7.37it/s, avg_rate=0.265 target=1.50]

[mon] in_sum=270.0 lif_sum=5900.0 v_max=0.0 vt_min=0.12555000185966492
[W] mean=0.525335 max=0.799988 min=0.250008
[enc] shape=(200, 1, 784) sum=180.0 max=1.0
[enc] per_t: min=0.0 max=4.0 t_argmax=25
[mon] in_sum=180.0 lif_sum=5300.0 v_max=0.0 vt_min=0.1311199963092804
[W] mean=0.525335 max=0.799988 min=0.250008


...

Train (N=60000, T=200):   0%| | 2/60000 [00:00<2:37:47,  6.34it/s, spikes=600.00 uniq=6 HHI=0.167 e≈

[W] mean=0.524986 max=0.799988 min=0.235741
[enc] shape=(200, 1, 784) sum=162.0 max=1.0
[enc] per_t: min=0.0 max=4.0 t_argmax=185


Train (N=60000, T=200):   0%| | 4/60000 [00:00<2:52:23,  5.80it/s, spikes=600.00 uniq=12 HHI=0.083 e

[W] mean=0.524893 max=0.799988 min=0.235741


Train (N=60000, T=200):  51%|▌| 30865/60000 [1:19:35<1:37:34,  4.98it/s, spikes=534.94 uniq=100 HHI=